# search-cost

**How much cheaper is informed search?** BFS, Dijkstra and A\* over the same
queries on the same graph, measured rather than asserted.

The project catalogue states the problem as: *"show how informed search (A\*)
expands fewer nodes than uninformed search (BFS/Dijkstra) while returning the
same answer."* This notebook is where that claim gets its numbers.

It produces three required outputs:

1. the **BFS-vs-Dijkstra-vs-A\* comparison table** on long-haul queries,
2. the **runtime-vs-input-size plot**, by narrowing the world network down a
   size series and re-timing the same queries at each size, and
3. the **measured growth exponents** for graph construction and for each
   search — the empirical counterpart to the Big-O write-up, taken from our
   own implementation rather than from the textbook.

The data is pinned by `experiment.toml` and verified on load: if a byte of the
snapshot changes, this notebook raises instead of quietly producing a different
answer.

## Setup

Only the first cell differs between Colab and a local checkout.

In [ ]:
# In Colab, clone the repository and install the package first:
#   !git clone https://github.com/dgwartney/traiectoria-optima.git
#   %pip install -q ./traiectoria-optima matplotlib
# Then open this notebook from the clone.
try:
    import flight_planner  # noqa: F401
except ModuleNotFoundError as error:
    raise SystemExit('install the package first -- see the comment above') from error

In [ ]:
import importlib.util
import math
import os
import platform
import statistics
import subprocess
import time
from pathlib import Path

from flight_planner.experiments import Experiment
from flight_planner.geo import haversine_heuristic
from flight_planner.pathfinding import (
    COST_HOPS,
    AStar,
    BFS,
    Dijkstra,
    ExpansionTrace,
)

import plots

# The notebook lives in the experiment directory, so the experiment is
# right here. Nothing resolves against a repository root.
experiment = Experiment.open(Path.cwd())
parameters = experiment.parameters

PAIRS = [tuple(pair) for pair in parameters['pairs']]
REPEATS = parameters['repeats']
PAIRS, REPEATS

## The data

Opening the snapshot re-hashes every file against the manifest. This experiment
runs on the **whole world network**, unnarrowed — the catalogue asks for
long-haul queries, and those only exist if the graph has long hauls in it.

In [ ]:
snapshot = experiment.snapshot
print(snapshot.snapshot_id, snapshot.criteria or '(no criteria: the full network)')

catalog = experiment.catalog()
world = catalog.planner()
print(f'{len(catalog.airports):,} airports / {len(catalog.routes):,} routes')

## The three algorithms

One heuristic, used by A\* throughout: great-circle distance to the goal. It
never overestimates the remaining distance — no route between two airports is
shorter than the straight line between them — which is what makes it
admissible, and therefore what makes A\*'s answer optimal rather than merely
fast.

In [ ]:
def algorithms():
    """A fresh instance of each algorithm, in a fixed reporting order."""
    return [
        ('BFS', BFS()),
        ('Dijkstra', Dijkstra()),
        # haversine_heuristic() is the project's one admissible heuristic;
        # hand-rolling one is how the wrong formula gets in. A fresh call
        # per algorithms() means a cold cache each time this is rebuilt.
        ('A*', AStar(haversine_heuristic())),
    ]


NAMES = [name for name, _ in algorithms()]
NAMES

## 1. What each search costs

`search_route` returns the route *and* the counters the search accumulated:

- **expanded** — vertices whose outgoing edges were examined. This is the
  number the project's claim is about.
- **pushed** — vertices placed on the frontier, counting repeats. Both weighted
  algorithms re-queue a vertex rather than repositioning its entry, so
  `pushed - expanded` is the work the lazy-deletion design throws away.
- **peak** — the largest the frontier ever got, measured against the O(V) space
  bound the complexity write-up claims.

Note that **cost is not one unit**: BFS counts hops, the other two count
kilometres. They are never summed or compared directly.

In [ ]:
comparison = []
for origin, destination in PAIRS:
    row = {'pair': f'{origin}-{destination}',
           'expanded': {}, 'pushed': {}, 'peak': {}, 'cost': {},
           'cost_unit': {}, 'legs': {}}
    for name, algorithm in algorithms():
        result = world.search_route(origin, destination, algorithm)
        row['expanded'][name] = result.nodes_expanded
        row['pushed'][name] = result.nodes_pushed
        row['peak'][name] = result.peak_frontier
        row['cost'][name] = result.cost
        # BFS's cost is hops and the others' kilometres, and they share one
        # 'cost' mapping. results.json outlives the prose that explains that,
        # so the unit travels with the number.
        row['cost_unit'][name] = result.unit
        row['legs'][name] = len(result.path)
    comparison.append(row)

header = f"{'pair':<10}{'algorithm':<10}{'expanded':>9}{'pushed':>8}{'peak':>7}{'cost':>13}{'legs':>6}"
print(header)
print('-' * len(header))
for row in comparison:
    for name in NAMES:
        unit = 'hops' if row['cost_unit'][name] == COST_HOPS else 'km'
        print(f"{row['pair']:<10}{name:<10}{row['expanded'][name]:>9,}"
              f"{row['pushed'][name]:>8,}{row['peak'][name]:>7,}"
              f"{row['cost'][name]:>10,.0f} {unit:<5}{row['legs'][name]:>4}")
    print()

### Do Dijkstra and A\* actually agree?

The claim has two halves, and the second is worthless without the first. If A\*
were merely fast it would be a worse algorithm, not a better one.

In [ ]:
for row in comparison:
    agree = abs(row['cost']['Dijkstra'] - row['cost']['A*']) < 1e-9
    ratio = row['expanded']['Dijkstra'] / max(row['expanded']['A*'], 1)
    print(f"{row['pair']:<10} same distance: {str(agree):<6} "
          f"A* expanded {ratio:,.0f}x fewer nodes")

In [ ]:
plots.nodes_expanded(comparison, '../../slides/images/nodes-expanded.png', mode='dark')
plots.nodes_expanded(comparison, '../../docs/images/nodes-expanded-light.png', mode='light')

## 2. Where each search looks

The counters say *how much*. An `ExpansionTrace` says *where* — it records each
expansion as it happens, so the two searches can be compared by shape rather
than by size.

In [ ]:
origin, destination = PAIRS[0]
for name, algorithm in algorithms():
    if name == 'BFS':
        continue
    trace = ExpansionTrace()
    world.search_route(origin, destination, algorithm, observer=trace)
    looked = ' '.join(airport.iata_code for airport in trace.order[:14])
    tail = ' ...' if len(trace.order) > 14 else ''
    print(f'{name:<10} {len(trace.order):>4} expansions: {looked}{tail}')

## 3. Runtime against input size

The rubric asks for at least one plot of time against input size. Sizes come
from narrowing the world catalog with the vocabulary the `Catalog` already has,
so no new code decides what "smaller" means.

Two decisions worth stating, because both are easy to get wrong:

- **The x-axis is V + E, not airports.** The two do not move together — the US
  large-airport network is 94 airports but 7,005 routes, while Delta's is 427
  airports and 2,170 routes. Ordering by airport count produces a curve that
  crosses itself. V + E is also the term in O((V + E) log V).
- **The same queries run at every size.** Every airport in `PAIRS` survives
  every narrowing below. If a pair vanished partway down the series, the curve
  would be comparing different questions at different sizes.
- **Graph construction is timed separately from search.** A search can stop
  early by reaching its goal; building the adjacency list cannot. So
  construction is the one operation in this project whose O(V + E) claim is
  directly testable, and it is measured here rather than cited.

### Which machine produced these numbers

Two kinds of result come out of this section, and they travel differently:

| Result | Portable? |
|---|---|
| Nodes expanded, and the growth exponents below | **Yes.** Properties of the algorithms. They reproduce on any machine. |
| Absolute milliseconds | **No.** A property of this machine, and only interpretable if the machine is named. |

So the run records what it ran on. Executing this notebook in Colab is the
cheapest way to make the timings verifiable by someone else: the hardware is
then a stated, selectable configuration rather than whatever laptop happened to
be to hand.

The workload is single-threaded pure Python — adjacency lists, a binary heap,
and arithmetic. There is no array math and no CUDA, so a **GPU runtime makes no
difference to any number here**; the CPU model and the Python version are what
matter.

In [ ]:
def environment():
    """Describe the machine, so a timing here is comparable with one elsewhere.

    Returns:
        Mapping of CPU model, core count, platform, Python version, and whether
        this is a Colab runtime.
    """
    cpu = platform.processor() or platform.machine()
    if platform.system() == 'Darwin':
        probe = subprocess.run(['sysctl', '-n', 'machdep.cpu.brand_string'],
                               capture_output=True, text=True)
        cpu = probe.stdout.strip() or cpu
    elif Path('/proc/cpuinfo').exists():          # Linux, including Colab
        for line in Path('/proc/cpuinfo').read_text().splitlines():
            if line.startswith('model name'):
                cpu = line.split(':', 1)[1].strip()
                break
    try:
        # find_spec raises rather than returning None when the parent package
        # 'google' is absent, which is the normal case off Colab.
        colab = importlib.util.find_spec('google.colab') is not None
    except ModuleNotFoundError:
        colab = False

    return {
        'cpu': cpu,
        'cores': os.cpu_count(),
        'platform': platform.platform(),
        'python': platform.python_version(),
        'colab': colab,
    }


machine = environment()
for key, value in machine.items():
    print(f'{key:<10}{value}')

In [ ]:
def narrow(base, spec):
    """Apply one size specification from experiment.toml to the catalog."""
    result = base
    if 'airline' in spec:
        result = result.airline(*spec['airline'])
    if 'country' in spec:
        result = result.country(*spec['country'])
    if 'airport_type' in spec:
        result = result.airport_type(*spec['airport_type'])
    return result


def median_ms(planner, algorithm):
    """Median query time in ms across PAIRS, after a discarded warm-up."""
    per_pair = []
    for origin, destination in PAIRS:
        planner.search_route(origin, destination, algorithm)   # warm-up
        runs = []
        for _ in range(REPEATS):
            started = time.perf_counter()
            # No observer while timing: measure the algorithm, not the watching.
            planner.search_route(origin, destination, algorithm)
            runs.append((time.perf_counter() - started) * 1000)
        per_pair.append(statistics.median(runs))
    return statistics.median(per_pair)


def median_build_ms(narrowed):
    """Median time to materialize a catalog into a graph, in ms.

    Timed the same way as a query -- one warm-up discarded, then REPEATS runs,
    median reported -- so the two columns are comparable.
    """
    narrowed.planner()   # warm-up, discarded
    runs = []
    for _ in range(REPEATS):
        started = time.perf_counter()
        narrowed.planner()
        runs.append((time.perf_counter() - started) * 1000)
    return statistics.median(runs)


In [ ]:
series = []
for spec in parameters['sizes']:
    narrowed = narrow(catalog, spec)
    planner = narrowed.planner()          # built once, outside the timed region
    vertices, edges = len(narrowed.airports), len(narrowed.routes)
    series.append({
        'label': spec['label'],
        'vertices': vertices,
        'edges': edges,
        'size': vertices + edges,
        'build_ms': median_build_ms(narrowed),
        'median_ms': {name: median_ms(planner, algorithm)
                      for name, algorithm in algorithms()},
    })

header = (f"{'narrowing':<18}{'V':>7}{'E':>9}{'V+E':>9}{'build':>10}"
          + ''.join(f'{n:>12}' for n in NAMES))
print(header)
print('-' * len(header))
for row in sorted(series, key=lambda r: r['size']):
    print(f"{row['label']:<18}{row['vertices']:>7,}{row['edges']:>9,}{row['size']:>9,}"
          f"{row['build_ms']:>10.2f}"
          + ''.join(f"{row['median_ms'][n]:>12.3f}" for n in NAMES))


In [ ]:
plots.runtime_vs_size(series, '../../slides/images/runtime.png', mode='dark')
plots.runtime_vs_size(series, '../../docs/images/runtime-light.png', mode='light')

## 4. Growth exponents, measured

The complexity write-up needs to describe *our* implementation, not the
textbook's. On a log-log axis a power law is a straight line whose slope is the
growth exponent, so fitting a line to each series gives the measured exponent
directly.

`r²` is reported alongside it and matters as much as the slope: it says whether
a power law describes the data at all. **A low `r²` here is a finding, not a
bad fit** — it means the cost of that search is not governed by graph size.


In [ ]:
def growth_exponent(sizes, times):
    """Least-squares slope and r^2 of log(time) against log(size).

    The slope is the measured growth exponent: 1.0 is linear in V + E, 0.0 is
    independent of it.
    """
    ordered = sorted(zip(sizes, times))
    log_x = [math.log(x) for x, _ in ordered]
    log_y = [math.log(y) for _, y in ordered]
    n = len(log_x)
    mean_x, mean_y = sum(log_x) / n, sum(log_y) / n
    sxy = sum((x - mean_x) * (y - mean_y) for x, y in zip(log_x, log_y))
    sxx = sum((x - mean_x) ** 2 for x in log_x)
    syy = sum((y - mean_y) ** 2 for y in log_y)
    return {'exponent': sxy / sxx, 'r_squared': (sxy ** 2) / (sxx * syy)}


sizes = [row['size'] for row in series]
scaling = {'build': growth_exponent(sizes, [row['build_ms'] for row in series])}
for name in NAMES:
    scaling[name] = growth_exponent(
        sizes, [row['median_ms'][name] for row in series])

print(f"{'series':<12}{'exponent':>10}{'r^2':>8}")
print('-' * 30)
for name, measured in scaling.items():
    print(f"{name:<12}{measured['exponent']:>10.2f}{measured['r_squared']:>8.3f}")


## Record

`experiment.record` writes `results.json` next to this notebook, carrying the
snapshot's identity, criteria and source commit alongside the numbers — so a
result can always be traced to the data that produced it.

In [ ]:
path = experiment.record({
    'comparison': comparison,
    'runtime_series': series,
    'scaling': scaling,
    'environment': machine,
    'agreement': [
        {'pair': row['pair'],
         'dijkstra_km': row['cost']['Dijkstra'],
         'astar_km': row['cost']['A*'],
         'expansion_ratio': row['expanded']['Dijkstra'] / max(row['expanded']['A*'], 1)}
        for row in comparison
    ],
}, catalog=catalog)
print(f'recorded -> {path}')

## What this shows

On the full world network, A\* returns **exactly** Dijkstra's distance on every
query while expanding orders of magnitude fewer nodes — the result the project
set out to demonstrate.

Two things worth carrying into the write-up because they complicate the simple
story:

- **BFS is not uniformly cheaper than Dijkstra.** It answers a different
  question (fewest hops, not shortest distance) and on some queries it expands
  *more* nodes than Dijkstra does while returning a longer route.
- **A\* pushes far more than it expands.** Its saving is in expansions — the
  expensive operation, since each one touches every outgoing edge — not in heap
  traffic. The `pushed` column is what makes that visible, and it is the honest
  answer to "is A\* really doing less work?"
- **Only construction shows its asymptotic exponent.** Graph construction
  measures ~1.0 with a near-perfect fit, which is O(V + E) confirmed on our own
  code. The three searches all measure *sublinear*, with fits that get worse as
  the algorithm gets smarter — because a goal-directed search stops when it
  arrives, so it never does the work its worst-case bound describes. A\*'s
  exponent is close to zero: it expands a handful of nodes whether the graph has
  2,000 edges or 66,000. That gap between the bound and the measurement is the
  substance of the complexity write-up, not an embarrassment to it.
